# Capstone2 Original Exploration Notebook

**Purpose:**  
This notebook preserves the original exploratory workflow used during the project. It includes early data loading, FIDE monthly-rating exploration, player filtering, sampling ideas, and intermediate checks that informed the final capstone pipeline.


In [1]:
# ============================================================
# Notebook overview
# Purpose:
# - Preserve the original exploratory work for the capstone project.
# - Document early data checks, filtering, sampling, and feature ideas.
# Note:
# - This notebook is not the frozen final modelling notebook.
# - Do not use this notebook to regenerate final report tables unless the
#   full report is intentionally being updated.
# ============================================================

import pandas as pd

pd.set_option("display.max_columns", None)      # show all columns
pd.set_option("display.width", 2000)            # increase print width
pd.set_option("display.max_colwidth", None)     # don't truncate long text
pd.set_option("display.max_rows", 50)           # optional: show more rows

In [2]:
# ============================================================
# Step 1: Import required libraries
# Purpose:
# - Load packages used for data loading, cleaning, analysis, and inspection.
# ============================================================

## 1. Environment setup and helper imports

This section loads Python libraries and sets up the notebook environment for exploratory data processing.


In [3]:
# ============================================================
# Step 2: Define or inspect file paths
# Purpose:
# - Set the working paths for raw/intermediate FIDE files used in exploration.
# ============================================================

from pathlib import Path
import pandas as pd

raw_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"


zip_files = sorted(raw_dir.rglob("*.zip"))

size_check = []

for zip_path in zip_files:
    size_check.append({
        "zip_file": zip_path.name,
        "size_bytes": zip_path.stat().st_size,
        "size_mb": round(zip_path.stat().st_size / (1024 * 1024), 2)
    })

size_df = pd.DataFrame(size_check)

duplicate_sizes = size_df[size_df.duplicated("size_bytes", keep=False)]
duplicate_sizes = duplicate_sizes.sort_values(["size_bytes", "zip_file"])

if duplicate_sizes.empty:
    print("No ZIP files have the same exact size.And hence every month's download was unique in name and data both")
else:
    print("ZIP files with the same exact size:")
    print(duplicate_sizes.to_string(index=False))

No ZIP files have the same exact size.And hence every month's download was unique in name and data both


In [4]:
# ============================================================
# Step 3: Load source data
# Purpose:
# - Read the available source file into a pandas DataFrame.
# - Inspect whether the file structure is usable for analysis.
# ============================================================

# Recursively read all ZIP files from your folder

import zipfile
import pandas as pd
from pathlib import Path

# Path to downloaded ZIP file
zip_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"

# Extract ZIP
extract_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

# Find extracted TXT file
txt_files = list(extract_dir.glob("*.txt"))
print("Text file:",txt_files)

txt_path = txt_files[0]
print("**Reading:", txt_path)

# Read FIDE fixed-width TXT file
df = pd.read_fwf(txt_path)

# View first rows
print("**Few Records:\n",df.head())
print("\n**Dataframe info:\n",df.info)
      

Text file: [PosixPath('/Users/arunkumar/Downloads/fide_standard_rating/standard_rating_list.txt')]
**Reading: /Users/arunkumar/Downloads/fide_standard_rating/standard_rating_list.txt
**Few Records:
    ID Number                   Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0  537001345   A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018  NaN
1   10245154  A B M Jobair, Hossain  BAN   M  NaN  NaN  NaN  NaN   1750    0  40   1998    i
2   25121731             A C J John  IND   M  NaN  NaN  NaN  NaN   1438    0  40   1987    i
3   35077023         A Chakravarthy  IND   M  NaN  NaN  NaN  NaN   1491    0  40   1986    i
4  537070436              A Darshil  IND   M  NaN  NaN  NaN  NaN   1419    0  40   2013  NaN

**Dataframe info:
 <bound method DataFrame.info of         ID Number                   Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0       537001345   A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018 

## 2. Initial FIDE data loading and inspection

This section reads early FIDE source files and inspects the structure, columns, and sample rows before formal cleaning.


In [5]:
# ============================================================
# Step 4: Inspect raw data structure
# Purpose:
# - Review columns, shape, sample records, and basic data quality.
# ============================================================

#Limit to Indian Players
india_df = df[df["Fed"] == "IND"]

print("**Few Records:\n",india_df.head())
print("\n Dataframe Shape:\n",india_df.shape)

**Few Records:
    ID Number                  Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0  537001345  A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018  NaN
2   25121731            A C J John  IND   M  NaN  NaN  NaN  NaN   1438    0  40   1987    i
3   35077023        A Chakravarthy  IND   M  NaN  NaN  NaN  NaN   1491    0  40   1986    i
4  537070436             A Darshil  IND   M  NaN  NaN  NaN  NaN   1419    0  40   2013  NaN
7   46601570          A G, Perumal  IND   M  NaN  NaN  NaN  NaN   1456    0  40   1983    i

 Dataframe Shape:
 (53408, 13)


In [6]:
# ============================================================
# Step 5: Clean or standardize key fields
# Purpose:
# - Convert identifiers, rating fields, birth year, dates/months, and numeric
#   values into formats that can be used later.
# ============================================================

#Limit to Youth Players only
youth_india_df = india_df[india_df["B-day"] >= 2008]

print("**Indian Youth Players, Few Records:\n",youth_india_df.head())
print("\n Dataframe Shape:\n",youth_india_df.shape)


**Indian Youth Players, Few Records:
     ID Number                     Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0   537001345     A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018  NaN
4   537070436                A Darshil  IND   M  NaN  NaN  NaN  NaN   1419    0  40   2013  NaN
25   33454710                A. HUDSON  IND   M  NaN  NaN  NaN  AIM   1644    0  40   2013    i
45   88105563  Aabhas Kumar Srivastava  IND   M  NaN  NaN  NaN  NaN   1946    7  40   2009  NaN
46   48765902            Aabhas Rajput  IND   M  NaN  NaN  NaN  NaN   1536    8  40   2015  NaN

 Dataframe Shape:
 (16194, 13)


In [7]:
# ============================================================
# Step 6: Filter relevant player records
# Purpose:
# - Apply exploratory filters such as federation, rating type, birth year,
#   or youth-player eligibility.
# ============================================================

print(df.columns)

Index(['ID Number', 'Name', 'Fed', 'Sex', 'Tit', 'WTit', 'OTit', 'FOA', 'APR26', 'Gms', 'K', 'B-day', 'Flag'], dtype='object')


In [8]:
# ============================================================
# Step 7: Check filtered data
# Purpose:
# - Verify row counts, unique players, missing values, and sample records.
# ============================================================

# Check column names first
print(df.columns, "\n")

# Filter by FIDE ID
player_df = df[df["ID Number"] == 48786365]

print(player_df.to_string(index=False))

#player_df = df[df["Name"] == "Ayaan Arora"]
#print(player_df.to_string(index=False))



Index(['ID Number', 'Name', 'Fed', 'Sex', 'Tit', 'WTit', 'OTit', 'FOA', 'APR26', 'Gms', 'K', 'B-day', 'Flag'], dtype='object') 

 ID Number            Name Fed Sex Tit WTit OTit FOA  APR26  Gms  K  B-day Flag
  48786365 Nova Ayer Juyal IND   M NaN  NaN  NaN NaN   1610   18 40   2016  NaN


## 3. Filtering and player-pool exploration

This section explores filters such as federation, birth year, youth criteria, and player availability.


In [9]:
# ============================================================
# Step 8: Explore Indian youth player pool
# Purpose:
# - Estimate the broad Indian youth FIDE-rated player pool used to define
#   the project population.
# ============================================================

import zipfile
import pandas as pd
from pathlib import Path
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

raw_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "raw"

zip_files = sorted(raw_dir.rglob("*.zip"))

print("ZIP files found:", len(zip_files))

all_months = []

# Month-rating column pattern: JAN24, FEB24, MAR25, APR26 etc.
month_col_pattern = re.compile(r"^(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\d{2}$")

for zip_path in zip_files:
    print("Reading:", zip_path.name)

    # Extract rating_month from filename like standard_2024_01.zip
    parts = zip_path.stem.split("_")
    year = parts[-2]
    month = parts[-1]
    rating_month = f"{year}-{month}"

    with zipfile.ZipFile(zip_path, "r") as z:
        txt_files = [name for name in z.namelist() if name.lower().endswith(".txt")]

        if not txt_files:
            print("No txt found in", zip_path.name)
            continue

        txt_file = txt_files[0]

        with z.open(txt_file) as f:
            temp_df = pd.read_fwf(f)

    # Clean column names
    temp_df.columns = temp_df.columns.str.strip()

    # Find the monthly rating column, e.g. JAN24, FEB24, APR26
    rating_cols = [col for col in temp_df.columns if month_col_pattern.match(str(col).strip())]

    if len(rating_cols) == 0:
        print("No rating month column found in:", zip_path.name)
        print(temp_df.columns)
        continue

    rating_col = rating_cols[0]

    # Rename monthly rating column to common name
    temp_df = temp_df.rename(columns={rating_col: "standard_rating"})

    # Add rating month
    temp_df["rating_month"] = rating_month
    temp_df["source_file"] = zip_path.name

    all_months.append(temp_df)

history_df = pd.concat(all_months, ignore_index=True)

print("\n**Combined shape:", history_df.shape)
print("\n**Few Recpords:", history_df.head(10).to_string(index=False))

ZIP files found: 28
Reading: standard_2024_01.zip
Reading: standard_2024_02.zip
Reading: standard_2024_03.zip
Reading: standard_2024_04.zip
Reading: standard_2024_05.zip
Reading: standard_2024_06.zip
Reading: standard_2024_07.zip
Reading: standard_2024_08.zip
Reading: standard_2024_09.zip
Reading: standard_2024_10.zip
Reading: standard_2024_11.zip
Reading: standard_2024_12.zip
Reading: standard_2025_01.zip
Reading: standard_2025_02.zip
Reading: standard_2025_03.zip
Reading: standard_2025_04.zip
Reading: standard_2025_05.zip
Reading: standard_2025_06.zip
Reading: standard_2025_07.zip
Reading: standard_2025_08.zip
Reading: standard_2025_09.zip
Reading: standard_2025_10.zip
Reading: standard_2025_11.zip
Reading: standard_2025_12.zip
Reading: standard_2026_01.zip
Reading: standard_2026_03.zip
Reading: standard_2026_04.zip
Reading: standard_2026_05.zip

**Combined shape: (13858699, 15)

**Few Recpords:  ID Number                     Name Fed Sex Tit WTit OTit FOA  standard_rating  Gms  K  B

In [10]:
# ============================================================
# Step 9: Create or test sampling logic
# Purpose:
# - Explore how a sample can be drawn from the eligible player pool.
# - Later notebooks formalize this into a fixed reproducible sample.
# ============================================================

#To check one player
player_history = history_df[
    history_df["ID Number"].astype(str).str.strip() == "25977890"
].sort_values("rating_month")

print("**One sample player:537070436\n",player_history.to_string(index=False))

**One sample player:537070436
  ID Number                   Name Fed Sex Tit WTit OTit FOA  standard_rating  Gms  K  B-day Flag rating_month          source_file
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1473   24 40   2011  NaN      2024-01 standard_2024_01.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1520    8 40   2011  NaN      2024-02 standard_2024_02.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1734   16 40   2011  NaN      2024-03 standard_2024_03.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1719    8 40   2011  NaN      2024-04 standard_2024_04.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1718   17 40   2011  NaN      2024-05 standard_2024_05.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1749   17 40   2011  NaN      2024-06 standard_2024_06.zip
  25977890          Amartya Kumar IND   M NaN  NaN  

In [11]:
# ============================================================
# Step 10: Validate sample characteristics
# Purpose:
# - Check rating bands, activity, and basic player composition of the sample.
# ============================================================

#To keep only useful columns
useful_cols = [
    "ID Number", "Name", "Fed", "Sex", "Tit",
    "standard_rating", "Gms", "K", "B-day",
    "rating_month", "source_file"
]

history_clean = history_df[useful_cols]

print(history_clean.head(20).to_string(index=False))

 ID Number                         Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
  10245154        A B M Jobair, Hossain BAN   M NaN             1583    0 40   1998      2024-01 standard_2024_01.zip
  25121731                   A C J John IND   M NaN             1063    0 40   1987      2024-01 standard_2024_01.zip
  35077023               A Chakravarthy IND   M NaN             1151    0 40   1986      2024-01 standard_2024_01.zip
  10207538             A E M, Doshtagir BAN   M NaN             1840    0 40   1974      2024-01 standard_2024_01.zip
  10680810     A hamed Ashraf, Abdallah EGY   M NaN             1728    0 40   2001      2024-01 standard_2024_01.zip
   5716365              A Hamid, Harman MAS   M NaN             1325    0 40   1970      2024-01 standard_2024_01.zip
  10206612                A K M, Sourab BAN   M NaN             1598    0 20      0      2024-01 standard_2024_01.zip
   5045886                A K, Kalshyan IND   M NaN     

In [12]:
# ============================================================
# Step 11: Build exploratory features
# Purpose:
# - Create early versions of variables such as rating_start, games played,
#   active months, inactivity, and rating gain.
# ============================================================

history_clean
# Filter by FIDE ID
player_df = history_clean[history_clean["ID Number"] == 25977890]

print(player_df.to_string(index=False))


 ID Number                   Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
  25977890          Amartya Kumar IND   M NaN             1473   24 40   2011      2024-01 standard_2024_01.zip
  25977890          Amartya Kumar IND   M NaN             1520    8 40   2011      2024-02 standard_2024_02.zip
  25977890          Amartya Kumar IND   M NaN             1734   16 40   2011      2024-03 standard_2024_03.zip
  25977890          Amartya Kumar IND   M NaN             1719    8 40   2011      2024-04 standard_2024_04.zip
  25977890          Amartya Kumar IND   M NaN             1718   17 40   2011      2024-05 standard_2024_05.zip
  25977890          Amartya Kumar IND   M NaN             1749   17 40   2011      2024-06 standard_2024_06.zip
  25977890          Amartya Kumar IND   M NaN             1749   17 40   2011      2024-07 standard_2024_07.zip
  25977890 Amartya Saumitra Gupta IND   M NaN             1686    7 40   2011      2024-08 standard_2024

## 4. Sampling exploration

This section experiments with player sampling logic. These sampling ideas informed the fixed 1,000-player sample later used in the final report.


In [13]:
# ============================================================
# Step 12: Inspect feature distributions
# Purpose:
# - Understand whether engineered variables have plausible ranges and
#   missing-value patterns.
# ============================================================

#Save all the data for retrival later
output_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "fide_standard_history_clean.csv"

history_clean.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: /Users/arunkumar/Downloads/fide_history/fide_standard_history_clean.csv


In [14]:
# ============================================================
# Step 13: Run sanity checks
# Purpose:
# - Confirm that key counts, joins, and filtering steps behave as expected.
# ============================================================

#read it directly from saved csv
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

csv_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "fide_standard_history_clean.csv"

history_df = pd.read_csv(csv_path)

print(history_df.shape)
print(history_df.head(10).to_string(index=False))

(13858699, 11)
 ID Number                     Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
  10245154    A B M Jobair, Hossain BAN   M NaN             1583    0 40   1998      2024-01 standard_2024_01.zip
  25121731               A C J John IND   M NaN             1063    0 40   1987      2024-01 standard_2024_01.zip
  35077023           A Chakravarthy IND   M NaN             1151    0 40   1986      2024-01 standard_2024_01.zip
  10207538         A E M, Doshtagir BAN   M NaN             1840    0 40   1974      2024-01 standard_2024_01.zip
  10680810 A hamed Ashraf, Abdallah EGY   M NaN             1728    0 40   2001      2024-01 standard_2024_01.zip
   5716365          A Hamid, Harman MAS   M NaN             1325    0 40   1970      2024-01 standard_2024_01.zip
  10206612            A K M, Sourab BAN   M NaN             1598    0 20      0      2024-01 standard_2024_01.zip
   5045886            A K, Kalshyan IND   M NaN             1671    0 20 

In [15]:
# ============================================================
# Step 14: Save exploratory outputs
# Purpose:
# - Export intermediate files created during exploration.
# - Later data-preparation notebooks should be treated as cleaner sources.
# ============================================================

history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()
history_df["rating_month"] = pd.to_datetime(history_df["rating_month"], format="%Y-%m")

history_df["standard_rating"] = pd.to_numeric(history_df["standard_rating"], errors="coerce")
history_df["Gms"] = pd.to_numeric(history_df["Gms"], errors="coerce")
history_df["K"] = pd.to_numeric(history_df["K"], errors="coerce")
history_df["B-day"] = pd.to_numeric(history_df["B-day"], errors="coerce")

In [16]:
# ============================================================
# Step 15: Additional exploratory checks
# Purpose:
# - Retain extra analysis used during project development.
# ============================================================

player_history = history_df[
    history_df["ID Number"] == "25977890"
].sort_values("rating_month")

print(player_history.to_string(index=False))

ID Number                   Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
 25977890          Amartya Kumar IND   M NaN             1473   24 40   2011   2024-01-01 standard_2024_01.zip
 25977890          Amartya Kumar IND   M NaN             1520    8 40   2011   2024-02-01 standard_2024_02.zip
 25977890          Amartya Kumar IND   M NaN             1734   16 40   2011   2024-03-01 standard_2024_03.zip
 25977890          Amartya Kumar IND   M NaN             1719    8 40   2011   2024-04-01 standard_2024_04.zip
 25977890          Amartya Kumar IND   M NaN             1718   17 40   2011   2024-05-01 standard_2024_05.zip
 25977890          Amartya Kumar IND   M NaN             1749   17 40   2011   2024-06-01 standard_2024_06.zip
 25977890          Amartya Kumar IND   M NaN             1749   17 40   2011   2024-07-01 standard_2024_07.zip
 25977890 Amartya Saumitra Gupta IND   M NaN             1686    7 40   2011   2024-08-01 standard_2024_08.zip
 

## 5. Early feature construction checks

This section explores candidate variables such as rating, activity, games played, months active, and future rating gain.


In [17]:
# ============================================================
# Exploratory code cell 16
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

india_youth = history_df[
    (history_df["Fed"] == "IND") &
    (history_df["B-day"] >= 2008)
].copy()

print(india_youth.shape)
print(india_youth.head(20).to_string(index=False))

(305876, 11)
ID Number                       Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
 88105563    Aabhas Kumar Srivastava IND   M NaN             1284   10 40   2009   2024-01-01 standard_2024_01.zip
 48706809                    Aabhash IND   M NaN             1361    0 40   2009   2024-01-01 standard_2024_01.zip
 25199080               Aadhav Ashok IND   M NaN             1005    0 40   2010   2024-01-01 standard_2024_01.zip
 25775600              Aadhesh Nambi IND   M NaN             1146    0 40   2009   2024-01-01 standard_2024_01.zip
 48726699               Aadhinya K R IND   F NaN             1097    0 40   2013   2024-01-01 standard_2024_01.zip
 33482926              Aadhityaa S R IND   M NaN             1118    0 40   2011   2024-01-01 standard_2024_01.zip
 25974068                Aadhya Jain IND   F NaN             1422    0 40   2011   2024-01-01 standard_2024_01.zip
 33476497 Aadhya Kalpeshbhai Bhavsar IND   F NaN             1095  

In [18]:
# Sanity Checks
import zipfile
from pathlib import Path
import pandas as pd

raw_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "raw"

zip_files = sorted(raw_dir.rglob("*.zip"))

zip_check = []

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        txt_files = [n for n in names if n.lower().endswith(".txt")]
        
        zip_check.append({
            "zip_file": zip_path.name,
            "zip_size_mb": round(zip_path.stat().st_size / (1024 * 1024), 2),
            "txt_files_inside": txt_files
        })

zip_check_df = pd.DataFrame(zip_check)

print(zip_check_df.to_string(index=False))

            zip_file  zip_size_mb           txt_files_inside
standard_2024_01.zip         9.82    [standard_jan24frl.txt]
standard_2024_02.zip         9.99    [standard_feb24frl.txt]
standard_2024_03.zip        10.02    [standard_mar24frl.txt]
standard_2024_04.zip        10.16    [standard_apr24frl.txt]
standard_2024_05.zip        10.28    [standard_may24frl.txt]
standard_2024_06.zip        10.40    [standard_jun24frl.txt]
standard_2024_07.zip        10.39    [standard_jul24frl.txt]
standard_2024_08.zip        10.49    [standard_aug24frl.txt]
standard_2024_09.zip        10.56    [standard_sep24frl.txt]
standard_2024_10.zip        10.61    [standard_oct24frl.txt]
standard_2024_11.zip        10.71    [standard_nov24frl.txt]
standard_2024_12.zip        10.79    [standard_dec24frl.txt]
standard_2025_01.zip        10.87    [standard_jan25frl.txt]
standard_2025_02.zip        10.98    [standard_feb25frl.txt]
standard_2025_03.zip        11.06    [standard_mar25frl.txt]
standard_2025_04.zip    

In [19]:
# Sanity Checks

import re
import zipfile
import pandas as pd
from pathlib import Path

raw_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "raw"
zip_files = sorted(raw_dir.rglob("*.zip"))

month_col_pattern = re.compile(r"^(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\d{2}$")

actual_month_check = []

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        txt_files = [n for n in z.namelist() if n.lower().endswith(".txt")]
        
        if not txt_files:
            actual_month_check.append({
                "zip_file": zip_path.name,
                "actual_rating_column": None,
                "status": "No TXT file"
            })
            continue
        
        with z.open(txt_files[0]) as f:
            temp_df = pd.read_fwf(f, nrows=5)
        
        temp_df.columns = temp_df.columns.str.strip()
        rating_cols = [c for c in temp_df.columns if month_col_pattern.match(str(c).strip())]
        
        actual_month_check.append({
            "zip_file": zip_path.name,
            "actual_rating_column": rating_cols[0] if rating_cols else None,
            "status": "OK" if rating_cols else "No month rating column found"
        })

actual_month_check_df = pd.DataFrame(actual_month_check)

print(actual_month_check_df.to_string(index=False))

            zip_file actual_rating_column status
standard_2024_01.zip                JAN24     OK
standard_2024_02.zip                FEB24     OK
standard_2024_03.zip                MAR24     OK
standard_2024_04.zip                APR24     OK
standard_2024_05.zip                MAY24     OK
standard_2024_06.zip                JUN24     OK
standard_2024_07.zip                JUL24     OK
standard_2024_08.zip                AUG24     OK
standard_2024_09.zip                SEP24     OK
standard_2024_10.zip                OCT24     OK
standard_2024_11.zip                NOV24     OK
standard_2024_12.zip                DEC24     OK
standard_2025_01.zip                JAN25     OK
standard_2025_02.zip                FEB25     OK
standard_2025_03.zip                MAR25     OK
standard_2025_04.zip                APR25     OK
standard_2025_05.zip                MAY25     OK
standard_2025_06.zip                JUN25     OK
standard_2025_07.zip                JUL25     OK
standard_2025_08.zip

In [20]:
# ============================================================
# Exploratory code cell 19
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "fide_standard_history_clean.csv"

history_df = pd.read_csv(csv_path)

history_df.columns = history_df.columns.str.strip()

history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()
history_df["rating_month"] = pd.to_datetime(history_df["rating_month"], format="%Y-%m")

history_df["standard_rating"] = pd.to_numeric(history_df["standard_rating"], errors="coerce")
history_df["Gms"] = pd.to_numeric(history_df["Gms"], errors="coerce")
history_df["K"] = pd.to_numeric(history_df["K"], errors="coerce")
history_df["B-day"] = pd.to_numeric(history_df["B-day"], errors="coerce")

# Treat 0 birth year as missing
history_df["Birth_Year_Clean"] = history_df["B-day"].replace(0, np.nan)

# Create profile URL
history_df["FIDE_Profile_URL"] = (
    "https://ratings.fide.com/profile/" + history_df["ID Number"]
)

# Function to get last non-null value
def last_valid(series):
    valid = series.dropna()
    if len(valid) == 0:
        return np.nan
    return valid.iloc[-1]

player_validation_df = (
    history_df
    .sort_values("rating_month")
    .groupby("ID Number", as_index=False)
    .agg(
        Name=("Name", "last"),
        Fed=("Fed", "last"),
        Sex=("Sex", "last"),
        Title=("Tit", "last"),
        Birth_Year=("Birth_Year_Clean", last_valid),
        Latest_Rating=("standard_rating", "last"),
        Latest_Month=("rating_month", "last"),
        Months_Available=("rating_month", "nunique"),
        FIDE_Profile_URL=("FIDE_Profile_URL", "last")
    )
)

print(player_validation_df.head(20).to_string(index=False))

ID Number                      Name Fed Sex Title  Birth_Year  Latest_Rating Latest_Month  Months_Available                          FIDE_Profile_URL
 10000011          Kabuye, Emmanuel UGA   M  None         NaN           2243   2026-05-01                28 https://ratings.fide.com/profile/10000011
 10000038            Matovu, George UGA   M  None         NaN           2225   2026-05-01                28 https://ratings.fide.com/profile/10000038
 10000046       Kamuhangire, Silver UGA   M  None         NaN           2190   2026-05-01                28 https://ratings.fide.com/profile/10000046
 10000054         Ssentongo, Edward UGA   M  None      1957.0           2250   2026-05-01                28 https://ratings.fide.com/profile/10000054
 10000062            Okoth, Joachim UGA   M  None      1950.0           2171   2026-05-01                28 https://ratings.fide.com/profile/10000062
 10000070            Anuari, Julius UGA   M  None      1982.0           1833   2026-05-01           

## 6. Intermediate validation and sanity checks

This section performs checks on row counts, missing values, unique players, and selected player records.


In [21]:
# ============================================================
# Exploratory code cell 20
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

fide_id = "25977890"

player_profile_url = f"https://ratings.fide.com/profile/{fide_id}"

print(player_profile_url)

https://ratings.fide.com/profile/25977890


### Unique player validation table from your monthly database

In [23]:
#create a unique player validation table from your monthly database

player_validation_df = (
    history_df
    .sort_values("rating_month")
    .groupby("ID Number", as_index=False)
    .agg(
        Name=("Name", "last"),
        Fed=("Fed", "last"),
        Sex=("Sex", "last"),
        Title=("Tit", "last"),
        Birth_Year=("B-day", "last"),
        Latest_Rating=("standard_rating", "last"),
        Latest_Month=("rating_month", "last"),
        Months_Available=("rating_month", "nunique"),
        FIDE_Profile_URL=("FIDE_Profile_URL", "last")
    )
)

print(player_validation_df.head(20).to_string(index=False))

ID Number                      Name Fed Sex Title  Birth_Year  Latest_Rating Latest_Month  Months_Available                          FIDE_Profile_URL
 10000011          Kabuye, Emmanuel UGA   M  None           0           2243   2026-05-01                28 https://ratings.fide.com/profile/10000011
 10000038            Matovu, George UGA   M  None           0           2225   2026-05-01                28 https://ratings.fide.com/profile/10000038
 10000046       Kamuhangire, Silver UGA   M  None           0           2190   2026-05-01                28 https://ratings.fide.com/profile/10000046
 10000054         Ssentongo, Edward UGA   M  None        1957           2250   2026-05-01                28 https://ratings.fide.com/profile/10000054
 10000062            Okoth, Joachim UGA   M  None        1950           2171   2026-05-01                28 https://ratings.fide.com/profile/10000062
 10000070            Anuari, Julius UGA   M  None        1982           1833   2026-05-01           

In [24]:
# ============================================================
# Exploratory code cell 23
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

missing_birth = player_validation_df[player_validation_df["Birth_Year"].isna()]

print(missing_birth.head(20).to_string(index=False))
print("Missing birth year count:", missing_birth.shape[0])

Empty DataFrame
Columns: [ID Number, Name, Fed, Sex, Title, Birth_Year, Latest_Rating, Latest_Month, Months_Available, FIDE_Profile_URL]
Index: []
Missing birth year count: 0


## 7. Output/export exploration

This section writes intermediate files or inspects outputs that were later formalized in the clean data-preparation pipeline.


In [25]:
# ============================================================
# Exploratory code cell 24
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

fed_counts = (
    player_validation_df[player_validation_df["Birth_Year"].isna()]
    .groupby("Fed")
    .agg(
        player_count=("ID Number", "nunique")
    )
    .reset_index()
    .sort_values("player_count", ascending=False)
)

print(fed_counts.to_string(index=False))

Empty DataFrame
Columns: [Fed, player_count]
Index: []


In [26]:
# ============================================================
# Exploratory code cell 25
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

india_youth_count = player_validation_df[
    (player_validation_df["Fed"] == "IND") &
    (player_validation_df["Birth_Year"].notna()) &
    (player_validation_df["Birth_Year"] >= 2008)
]["ID Number"].nunique()

print("Unique Indian youth players:", india_youth_count)

Unique Indian youth players: 18753


#### Chess-Results will be used to extract tournament participation and performance variables, including scores, rankings, opponent strength, and event history.

In [28]:
#pip install pandas requests beautifulsoup4 lxml

In [29]:
# ============================================================
# Exploratory code cell 28
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
import re
from pathlib import Path
import time

In [30]:
# ============================================================
# Exploratory code cell 29
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def make_final_ranking_url(url):
    """
    Converts a Chess-Results tournament URL to final ranking page if possible.
    Usually art=9 gives final ranking / standings.
    """
    parsed = urlparse(url)
    query = parse_qs(parsed.query)

    query["art"] = ["9"]     # final ranking / standings
    query["lan"] = ["1"]     # English language where available

    new_query = urlencode(query, doseq=True)

    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        parsed.params,
        new_query,
        parsed.fragment
    ))

In [31]:
# ============================================================
# Exploratory code cell 30
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def get_tournament_id(url):
    parsed = urlparse(url)
    query = parse_qs(parsed.query)

    if "tnr" in query:
        return query["tnr"][0]
    return None

In [32]:
# ============================================================
# Exploratory code cell 31
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def read_chess_results_tables(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    html = response.text

    tables = pd.read_html(html)

    return tables, html

In [33]:
# ============================================================
# Exploratory code cell 32
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def find_ranking_table(tables):
    """
    Finds the table most likely to be the tournament ranking table.
    Looks for common Chess-Results columns such as Rk., Name, FED, Rtg, Pts.
    """

    candidate_tables = []

    for i, table in enumerate(tables):
        df = table.copy()
        df.columns = [str(c).strip() for c in df.columns]

        cols = set(df.columns)

        score = 0

        for possible_col in ["Rk.", "Rk", "Rank"]:
            if possible_col in cols:
                score += 2

        for possible_col in ["Name", "NameName"]:
            if possible_col in cols:
                score += 2

        for possible_col in ["FED", "Fed", "Federation"]:
            if possible_col in cols:
                score += 1

        for possible_col in ["Rtg", "Rating"]:
            if possible_col in cols:
                score += 1

        for possible_col in ["Pts.", "Pts", "Points"]:
            if possible_col in cols:
                score += 2

        if score >= 4:
            candidate_tables.append((score, i, df))

    if not candidate_tables:
        raise ValueError("No likely ranking table found.")

    candidate_tables = sorted(candidate_tables, reverse=True, key=lambda x: x[0])

    return candidate_tables[0][2]

In [34]:
# ============================================================
# Exploratory code cell 33
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def clean_chess_results_columns(df):
    df = df.copy()

    df.columns = [str(c).strip() for c in df.columns]

    rename_map = {
        "Rk.": "rank",
        "Rk": "rank",
        "Rank": "rank",

        "Name": "player_name",

        "FED": "fed",
        "Fed": "fed",

        "Rtg": "rating",
        "Rating": "rating",

        "Pts.": "points",
        "Pts": "points",
        "Points": "points",

        "Rp": "performance_rating",
        "Perf": "performance_rating",
        "Performance": "performance_rating",

        "ID": "fide_id",
        "FIDE ID": "fide_id",
        "FIDEID": "fide_id",
        "ID-No.": "fide_id",
        "ID-No": "fide_id"
    }

    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})

    return df

In [35]:
# ============================================================
# Exploratory code cell 34
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def extract_chess_results_tournament(url):
    final_url = make_final_ranking_url(url)
    tournament_id = get_tournament_id(final_url)

    print("Reading:", final_url)

    tables, html = read_chess_results_tables(final_url)

    ranking_df = find_ranking_table(tables)
    ranking_df = clean_chess_results_columns(ranking_df)

    ranking_df["tournament_id"] = tournament_id
    ranking_df["tournament_url"] = final_url

    return ranking_df

In [36]:
# ============================================================
# Exploratory code cell 35
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

tournament_urls = [
    "https://s1.chess-results.com/tnr1264363.aspx?lan=1&SNode=S0",
    "https://s3.chess-results.com/tnr1134410.aspx?SNode=S0"
]

all_tournaments = []

for url in tournament_urls:
    try:
        df_event = extract_chess_results_tournament(url)
        all_tournaments.append(df_event)

        time.sleep(2)  # polite delay

    except Exception as e:
        print("Failed:", url)
        print("Reason:", e)

chess_results_df = pd.concat(all_tournaments, ignore_index=True)

print(chess_results_df.head(20).to_string(index=False))
print(chess_results_df.shape)

Reading: https://s1.chess-results.com/tnr1264363.aspx?lan=1&SNode=S0&art=9


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_66265/699151273.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Failed: https://s1.chess-results.com/tnr1264363.aspx?lan=1&SNode=S0
Reason: No likely ranking table found.
Reading: https://s3.chess-results.com/tnr1134410.aspx?SNode=S0&art=9&lan=1
Failed: https://s3.chess-results.com/tnr1134410.aspx?SNode=S0
Reason: No likely ranking table found.


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_66265/699151273.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


ValueError: No objects to concatenate

In [ ]:
# ============================================================
# Exploratory code cell 36
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import requests
from io import StringIO
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

def make_url(url, art=9):
    parsed = urlparse(url)
    query = parse_qs(parsed.query)

    query["lan"] = ["1"]
    query["art"] = [str(art)]

    new_query = urlencode(query, doseq=True)

    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        parsed.params,
        new_query,
        parsed.fragment
    ))

def debug_chess_results_tables(url, art=9):
    final_url = make_url(url, art=art)
    print("Reading:", final_url)

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9"
    }

    response = requests.get(final_url, headers=headers, timeout=30)
    print("HTTP status:", response.status_code)
    response.raise_for_status()

    html = response.text

    tables = pd.read_html(StringIO(html))

    print("Number of tables found:", len(tables))

    for i, table in enumerate(tables):
        print("\n" + "="*100)
        print("TABLE:", i)
        print("Shape:", table.shape)
        print("Columns:")
        print(table.columns.tolist())
        print("Head:")
        print(table.head(10).to_string(index=False))

    return tables

In [ ]:
# ============================================================
# Exploratory code cell 37
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

url = "https://s1.chess-results.com/tnr1264363.aspx?lan=1&SNode=S0"

tables = debug_chess_results_tables(url, art=9)

In [ ]:
# ============================================================
# Exploratory code cell 38
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

tables_art4 = debug_chess_results_tables(url, art=4)

In [ ]:
# ============================================================
# Exploratory code cell 39
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

url2 = "https://s3.chess-results.com/tnr1134410.aspx?SNode=S0"

tables2 = debug_chess_results_tables(url2, art=9)
tables2_art4 = debug_chess_results_tables(url2, art=4)

In [ ]:
# ============================================================
# Exploratory code cell 40
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import requests
from io import StringIO
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
import time

def make_chess_results_url(url, art=9):
    parsed = urlparse(url)
    query = parse_qs(parsed.query)

    query["lan"] = ["1"]
    query["art"] = [str(art)]

    new_query = urlencode(query, doseq=True)

    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        parsed.params,
        new_query,
        parsed.fragment
    ))

def get_tournament_id(url):
    parsed = urlparse(url)
    file_name = parsed.path.split("/")[-1]
    return file_name.replace(".aspx", "")

def flatten_columns(df):
    df = df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(x) for x in col if str(x) != "nan"]).strip()
            for col in df.columns
        ]
    else:
        df.columns = [str(c).strip() for c in df.columns]

    return df

def clean_possible_header(df):
    """
    Sometimes Chess-Results tables are read with wrong headers.
    If columns are 0,1,2,... and first row contains Rk/Name/Rtg/Pts, use first row as header.
    """
    df = df.copy()
    df = flatten_columns(df)

    first_row_text = " ".join(df.iloc[0].astype(str).tolist()).lower() if len(df) > 0 else ""

    if (
        ("name" in first_row_text or "player" in first_row_text) and
        ("rk" in first_row_text or "rank" in first_row_text or "pts" in first_row_text)
    ):
        new_cols = df.iloc[0].astype(str).str.strip().tolist()
        df = df.iloc[1:].copy()
        df.columns = new_cols

    df.columns = [str(c).strip() for c in df.columns]

    return df

def score_table_as_ranking(df):
    df = clean_possible_header(df)

    columns_text = " ".join([str(c).lower() for c in df.columns])
    sample_text = " ".join(df.head(5).astype(str).fillna("").values.flatten()).lower()

    text = columns_text + " " + sample_text

    score = 0

    if "name" in text:
        score += 3
    if "fed" in text or "federation" in text:
        score += 2
    if "rtg" in text or "rating" in text:
        score += 2
    if "pts" in text or "points" in text:
        score += 3
    if "rk" in text or "rank" in text:
        score += 2
    if df.shape[0] >= 10:
        score += 2
    if df.shape[1] >= 5:
        score += 2

    return score, df

def find_best_ranking_table(tables):
    candidates = []

    for i, table in enumerate(tables):
        score, cleaned = score_table_as_ranking(table)
        candidates.append((score, i, cleaned))

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    print("Table scores:")
    for score, i, cleaned in candidates:
        print(f"Table {i}: score={score}, shape={cleaned.shape}, columns={cleaned.columns.tolist()}")

    best_score, best_i, best_df = candidates[0]

    if best_score < 5:
        raise ValueError("No likely ranking table found even with flexible scoring.")

    print("Selected table:", best_i)

    return best_df

def clean_chess_results_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    rename_map = {
        "Rk.": "rank",
        "Rk": "rank",
        "Rank": "rank",
        "Ranking": "rank",

        "Name": "player_name",
        "Name Name": "player_name",

        "FED": "fed",
        "Fed": "fed",
        "Federation": "fed",

        "Rtg": "rating",
        "Rtg.": "rating",
        "Rating": "rating",

        "Pts.": "points",
        "Pts": "points",
        "Points": "points",

        "Rp": "performance_rating",
        "Perf": "performance_rating",
        "Performance": "performance_rating",

        "ID": "fide_id",
        "FIDE ID": "fide_id",
        "FIDEID": "fide_id",
        "ID-No.": "fide_id",
        "ID-No": "fide_id",
        "Fide-No": "fide_id",
        "FideNo": "fide_id",
    }

    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})

    return df

def extract_chess_results_tournament(url):
    tournament_id = get_tournament_id(url)

    # Try multiple art values
    art_values = [9, 4, 1]

    last_error = None

    for art in art_values:
        try:
            final_url = make_chess_results_url(url, art=art)
            print("\nReading:", final_url)

            headers = {
                "User-Agent": "Mozilla/5.0",
                "Accept-Language": "en-US,en;q=0.9"
            }

            response = requests.get(final_url, headers=headers, timeout=30)
            print("HTTP status:", response.status_code)
            response.raise_for_status()

            tables = pd.read_html(StringIO(response.text))
            print("Tables found:", len(tables))

            ranking_df = find_best_ranking_table(tables)
            ranking_df = clean_chess_results_columns(ranking_df)

            ranking_df["tournament_id"] = tournament_id
            ranking_df["tournament_url"] = final_url
            ranking_df["art_used"] = art

            return ranking_df

        except Exception as e:
            print("Failed with art =", art, "| Reason:", repr(e))
            last_error = e

    raise last_error

In [ ]:
# ============================================================
# Exploratory code cell 41
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

tournament_urls = [
    "https://s1.chess-results.com/tnr1264363.aspx?lan=1&SNode=S0",
    "https://s3.chess-results.com/tnr1134410.aspx?SNode=S0"
]

all_tournaments = []

for url in tournament_urls:
    try:
        df_event = extract_chess_results_tournament(url)
        all_tournaments.append(df_event)

        print("Success:", url)
        print("Rows extracted:", df_event.shape[0])
        print(df_event.head(10).to_string(index=False))

        time.sleep(2)

    except Exception as e:
        print("Failed completely:", url)
        print("Reason:", repr(e))

if len(all_tournaments) == 0:
    print("No tournament data extracted. Do not run pd.concat because list is empty.")
else:
    chess_results_df = pd.concat(all_tournaments, ignore_index=True)
    print(chess_results_df.head(20).to_string(index=False))
    print(chess_results_df.shape)

In [ ]:
# ============================================================
# Exploratory code cell 42
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

chess_results_df

In [ ]:
# ============================================================
# Exploratory code cell 43
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import re
import math
from pathlib import Path

df = chess_results_df.copy()
df = df.loc[:, ~df.columns.duplicated()].copy()

clean_rows = []

def is_number(x):
    try:
        if pd.isna(x):
            return False
        num = float(x)
        if math.isnan(num):
            return False
        return True
    except:
        return False

def is_valid_fed(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    return bool(re.match(r"^[A-Z]{3}$", s))

for idx, row in df.iterrows():
    values = row.tolist()

    fed_positions = []
    for i, v in enumerate(values):
        if is_valid_fed(v):
            fed_positions.append(i)

    if not fed_positions:
        continue

    for fed_pos in fed_positions:
        if fed_pos < 2:
            continue

        fed = str(values[fed_pos]).strip()

        player_name = values[fed_pos - 1]
        if pd.isna(player_name):
            continue

        player_name = str(player_name).strip()

        if player_name == "" or player_name.lower() in ["nan", "none"]:
            continue

        title = values[fed_pos - 2] if fed_pos - 2 >= 0 else None
        title = None if pd.isna(title) else str(title).strip()

        # Find real rank by looking backward before title/name/fed
        rank = None

        for j in range(fed_pos - 3, -1, -1):
            if is_number(values[j]):
                rank_float = float(values[j])

                # Skip non-positive values
                if rank_float <= 0:
                    continue

                rank = int(rank_float)
                break

        if rank is None:
            continue

        after_fed = values[fed_pos + 1:]

        clean_rows.append({
            "original_index": idx,
            "rank": rank,
            "title": title,
            "player_name": player_name,
            "fed": fed,
            "raw_after_fed": after_fed
        })

clean_base = pd.DataFrame(clean_rows)

print(clean_base.head(30).to_string(index=False))
print(clean_base.shape)

In [ ]:
# ============================================================
# Exploratory code cell 44
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def find_tnr(row):
    for v in row:
        s = str(v)
        m = re.search(r"tnr\d+", s)
        if m:
            return m.group(0)
    return None

def find_url(row):
    for v in row:
        s = str(v)
        if "chess-results.com" in s:
            return s
    return None

df_meta = df.copy()
df_meta["tournament_id_detected"] = df_meta.apply(lambda r: find_tnr(r.tolist()), axis=1)
df_meta["tournament_url_detected"] = df_meta.apply(lambda r: find_url(r.tolist()), axis=1)

clean_base["tournament_id"] = clean_base["original_index"].map(df_meta["tournament_id_detected"])
clean_base["tournament_url"] = clean_base["original_index"].map(df_meta["tournament_url_detected"])

clean_base = clean_base[
    ["tournament_id", "tournament_url", "rank", "title", "player_name", "fed", "raw_after_fed"]
]

print(clean_base.head(30).to_string(index=False))
print(clean_base.shape)

In [ ]:
# ============================================================
# Exploratory code cell 45
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def parse_after_fed(raw):
    raw = list(raw)

    if len(raw) >= 4:
        rounds = raw[:-4]
        points = raw[-4]
        tb1 = raw[-3]
        tb2 = raw[-2]
        tb3 = raw[-1]
    else:
        rounds = raw
        points = None
        tb1 = None
        tb2 = None
        tb3 = None

    return pd.Series({
        "round_results": rounds,
        "points": points,
        "TB1": tb1,
        "TB2": tb2,
        "TB3": tb3
    })

parsed = clean_base["raw_after_fed"].apply(parse_after_fed)

clean_base = pd.concat(
    [clean_base.drop(columns=["raw_after_fed"]), parsed],
    axis=1
)

for col in ["points", "TB1", "TB2", "TB3"]:
    clean_base[col] = pd.to_numeric(clean_base[col], errors="coerce")

print(clean_base.head(30).to_string(index=False))
print(clean_base.shape)

In [ ]:
# Try to identify tournament id and url from each original row
def find_tnr(row):
    for v in row:
        s = str(v)
        m = re.search(r"tnr\d+", s)
        if m:
            return m.group(0)
    return None

def find_url(row):
    for v in row:
        s = str(v)
        if "chess-results.com" in s:
            return s
    return None

df_meta = df.copy()
df_meta["tournament_id_detected"] = df_meta.apply(lambda r: find_tnr(r.tolist()), axis=1)
df_meta["tournament_url_detected"] = df_meta.apply(lambda r: find_url(r.tolist()), axis=1)

clean_base["tournament_id"] = clean_base["original_index"].map(df_meta["tournament_id_detected"])
clean_base["tournament_url"] = clean_base["original_index"].map(df_meta["tournament_url_detected"])

# Reorder columns
clean_base = clean_base[
    ["tournament_id", "tournament_url", "rank", "title", "player_name", "fed", "raw_after_fed"]
]

print(clean_base.head(30).to_string(index=False))
print(clean_base.shape)

In [ ]:
# ============================================================
# Exploratory code cell 48
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

output_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "chess_results_tournament_data.csv"

chess_results_df.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
# ============================================================
# Exploratory code cell 49
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

fide_id = "537070436"

# Some Chess-Results pages may not include FIDE ID.
# If fide_id exists, use it.
if "fide_id" in chess_results_df.columns:
    chess_results_df["fide_id"] = chess_results_df["fide_id"].astype(str).str.strip()

    player_events = chess_results_df[
        chess_results_df["fide_id"] == fide_id
    ].copy()

    print(player_events.to_string(index=False))
else:
    print("FIDE ID column not found in this Chess-Results table.")

In [ ]:
# ============================================================
# Exploratory code cell 50
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

df = chess_results_df.copy()

# Clean numeric columns
for col in ["rank", "rating", "points", "performance_rating"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Create features by FIDE ID if available
if "fide_id" in df.columns:
    player_tournament_features = (
        df.groupby("fide_id", as_index=False)
        .agg(
            tournaments_played=("tournament_id", "nunique"),
            avg_points=("points", "mean"),
            best_rank=("rank", "min"),
            avg_player_rating=("rating", "mean"),
            best_performance_rating=("performance_rating", "max")
        )
    )

    print(player_tournament_features.head(20).to_string(index=False))
else:
    print("Cannot create FIDE-ID-level features because fide_id column is missing.")

In [ ]:
# ============================================================
# Exploratory code cell 51
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()

if "fide_id" in player_tournament_features.columns:
    player_tournament_features["fide_id"] = player_tournament_features["fide_id"].astype(str).str.strip()

    merged_df = history_df.merge(
        player_tournament_features,
        left_on="ID Number",
        right_on="fide_id",
        how="left"
    )

    print(merged_df.head(20).to_string(index=False))

In [ ]:
#####################Yahan de naya hai

In [ ]:
# ============================================================
# Exploratory code cell 53
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import requests
import re
import math
import time
from io import StringIO
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
from pathlib import Path
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

tournament_urls = [
    "https://s1.chess-results.com/tnr1264363.aspx?lan=1&SNode=S0",
    "https://s3.chess-results.com/tnr1134410.aspx?SNode=S0"
]

In [ ]:
# handles your two URLs, removes junk rows, extracts real player rows, fixes half-points like 85 → 8.5, and saves a clean CSV

def make_chess_results_url(url, art):
    parsed = urlparse(url)
    query = parse_qs(parsed.query)

    query["lan"] = ["1"]
    query["art"] = [str(art)]

    new_query = urlencode(query, doseq=True)

    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        parsed.params,
        new_query,
        parsed.fragment
    ))


def get_tournament_id(url):
    parsed = urlparse(url)
    filename = parsed.path.split("/")[-1]
    return filename.replace(".aspx", "")


def is_number(x):
    try:
        if pd.isna(x):
            return False
        num = float(x)
        if math.isnan(num):
            return False
        return True
    except Exception:
        return False


def is_valid_fed(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    return bool(re.match(r"^[A-Z]{3}$", s))


def fix_chess_decimal(value, max_reasonable=None):
    """
    Fix Chess-Results half-point values sometimes read incorrectly:
    85  -> 8.5
    655 -> 65.5
    415 -> 41.5
    """
    if pd.isna(value):
        return np.nan

    value = float(value)

    if max_reasonable is not None:
        if value > max_reasonable and value / 10 <= max_reasonable:
            return value / 10

    if value >= 100 and value % 5 == 0:
        return value / 10

    return value

In [ ]:
# ============================================================
# Exploratory code cell 55
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def extract_one_chess_results_tournament(url):
    tournament_id = get_tournament_id(url)

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9"
    }

    all_rows = []

    # Try multiple Chess-Results page views
    # art=9 often final ranking; art=4 often crosstable/ranking table
    for art in [9, 4, 1]:
        final_url = make_chess_results_url(url, art=art)
        print(f"\nReading: {final_url}")

        try:
            response = requests.get(final_url, headers=headers, timeout=30)
            print("HTTP status:", response.status_code)
            response.raise_for_status()

            tables = pd.read_html(StringIO(response.text))
            print("Tables found:", len(tables))

        except Exception as e:
            print("Failed reading art =", art, "| Reason:", repr(e))
            continue

        for table_index, table in enumerate(tables):
            df = table.copy()

            # Flatten multi-index columns if needed
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [
                    " ".join([str(x) for x in col if str(x) != "nan"]).strip()
                    for col in df.columns
                ]
            else:
                df.columns = [str(c).strip() for c in df.columns]

            df = df.loc[:, ~df.columns.duplicated()].copy()

            for idx, row in df.iterrows():
                values = row.tolist()

                # Find federation positions like IND, LTU, GEO, etc.
                fed_positions = [
                    i for i, v in enumerate(values)
                    if is_valid_fed(v)
                ]

                if not fed_positions:
                    continue

                for fed_pos in fed_positions:
                    if fed_pos < 2:
                        continue

                    fed = str(values[fed_pos]).strip()

                    player_name = values[fed_pos - 1]
                    if pd.isna(player_name):
                        continue

                    player_name = str(player_name).strip()

                    if player_name == "" or player_name.lower() in ["nan", "none"]:
                        continue

                    title = values[fed_pos - 2] if fed_pos - 2 >= 0 else None
                    title = None if pd.isna(title) else str(title).strip()

                    # Find rank by looking backward before title/name/fed
                    rank = None
                    for j in range(fed_pos - 3, -1, -1):
                        if is_number(values[j]):
                            rank_float = float(values[j])
                            if rank_float <= 0:
                                continue
                            rank = int(rank_float)
                            break

                    if rank is None:
                        continue

                    after_fed = values[fed_pos + 1:]

                    # Need enough values after federation:
                    # round results + points + tie-breaks
                    if len(after_fed) < 5:
                        continue

                    # Usually last 4 values are points, TB1, TB2, TB3
                    round_results = after_fed[:-4]
                    points = after_fed[-4]
                    tb1 = after_fed[-3]
                    tb2 = after_fed[-2]
                    tb3 = after_fed[-1]

                    num_rounds = len(round_results)

                    points = fix_chess_decimal(points, max_reasonable=num_rounds)
                    tb1 = fix_chess_decimal(tb1)
                    tb2 = fix_chess_decimal(tb2)
                    tb3 = fix_chess_decimal(tb3)

                    all_rows.append({
                        "tournament_id": tournament_id,
                        "tournament_url": final_url,
                        "art_used": art,
                        "table_index": table_index,
                        "rank": rank,
                        "title": title,
                        "player_name": player_name,
                        "fed": fed,
                        "num_rounds": num_rounds,
                        "round_results": round_results,
                        "points": points,
                        "TB1": tb1,
                        "TB2": tb2,
                        "TB3": tb3
                    })

        # If this art view produced player rows, stop trying other art values
        if len(all_rows) > 0:
            break

    result_df = pd.DataFrame(all_rows)

    if result_df.empty:
        print("No player rows extracted for:", url)
        return result_df

    # Remove duplicates caused by multiple table parsing
    result_df = result_df.drop_duplicates(
        subset=["tournament_id", "rank", "player_name", "fed"],
        keep="first"
    )

    result_df = result_df.sort_values(["tournament_id", "rank"]).reset_index(drop=True)

    return result_df

In [ ]:
# ============================================================
# Exploratory code cell 56
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

all_tournaments = []

for url in tournament_urls:
    df_event = extract_one_chess_results_tournament(url)

    if not df_event.empty:
        all_tournaments.append(df_event)
        print("Extracted rows:", df_event.shape[0])
        print(df_event.head(10).to_string(index=False))
    else:
        print("No rows extracted from:", url)

    time.sleep(2)

if len(all_tournaments) == 0:
    chess_results_clean = pd.DataFrame()
    print("No tournament data extracted.")
else:
    chess_results_clean = pd.concat(all_tournaments, ignore_index=True)
    chess_results_clean = chess_results_clean.sort_values(
        ["tournament_id", "rank"]
    ).reset_index(drop=True)

    print("\nFinal combined shape:", chess_results_clean.shape)
    print(chess_results_clean.head(30).to_string(index=False))

In [ ]:
# ============================================================
# Exploratory code cell 57
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

chess_results_clean

In [ ]:
# ============================================================
# Exploratory code cell 58
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

india_players = chess_results_clean[
    chess_results_clean["fed"] == "IND"
].copy()

print(india_players.to_string(index=False))
print("Indian players found:", india_players.shape[0])

In [ ]:
# ============================================================
# Exploratory code cell 59
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

output_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "chess_results_clean.csv"

chess_results_clean.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
# ============================================================
# Exploratory code cell 60
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

india_output_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "chess_results_india_players.csv"

india_players.to_csv(india_output_path, index=False)

print("Saved:", india_output_path)

##### create sample of 1,000 Indian youth players


In [ ]:
# ============================================================
# Exploratory code cell 62
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

csv_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "fide_standard_history_clean.csv"

history_df = pd.read_csv(csv_path)

history_df.columns = history_df.columns.str.strip()

history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()
history_df["rating_month"] = pd.to_datetime(history_df["rating_month"], format="%Y-%m")

history_df["standard_rating"] = pd.to_numeric(history_df["standard_rating"], errors="coerce")
history_df["Gms"] = pd.to_numeric(history_df["Gms"], errors="coerce").fillna(0)
history_df["K"] = pd.to_numeric(history_df["K"], errors="coerce")
history_df["B-day"] = pd.to_numeric(history_df["B-day"], errors="coerce")

# Treat 0 birth year as missing
history_df["Birth_Year"] = history_df["B-day"].replace(0, np.nan)

print(history_df.shape)
print(history_df.head().to_string(index=False))

In [ ]:
#Filter Indian youth players

#Let us define youth as U18 in 2026, meaning birth year 2008 or later.
india_youth_history = history_df[
    (history_df["Fed"] == "IND") &
    (history_df["Birth_Year"].notna()) &
    (history_df["Birth_Year"] >= 2008)
].copy()

print("Indian youth player-month records:", india_youth_history.shape)
print("Unique Indian youth players:", india_youth_history["ID Number"].nunique())

In [ ]:
# Build 12-month feature dataset

# Use Apr 2025 → Apr 2026 as the 12-month prediction/evaluation window.

start_month = pd.to_datetime("2025-04")
end_month = pd.to_datetime("2026-04")

period_df = india_youth_history[
    (india_youth_history["rating_month"] >= start_month) &
    (india_youth_history["rating_month"] <= end_month)
].copy()

print(period_df["rating_month"].min(), period_df["rating_month"].max())
print(period_df.shape)
print("Unique players in period:", period_df["ID Number"].nunique())

#### Create player-level features

In [ ]:
# ============================================================
# Exploratory code cell 66
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def first_valid(series):
    valid = series.dropna()
    return valid.iloc[0] if len(valid) > 0 else np.nan

def last_valid(series):
    valid = series.dropna()
    return valid.iloc[-1] if len(valid) > 0 else np.nan

player_features = (
    period_df
    .sort_values(["ID Number", "rating_month"])
    .groupby("ID Number", as_index=False)
    .agg(
        Name=("Name", "last"),
        Fed=("Fed", "last"),
        Sex=("Sex", "last"),
        Title=("Tit", "last"),
        Birth_Year=("Birth_Year", last_valid),

        rating_start=("standard_rating", first_valid),
        rating_end=("standard_rating", last_valid),

        months_available=("rating_month", "nunique"),
        total_games_12m=("Gms", "sum"),
        avg_monthly_games=("Gms", "mean"),
        max_monthly_games=("Gms", "max"),
        active_months_12m=("Gms", lambda x: (x > 0).sum()),
        inactive_months_12m=("Gms", lambda x: (x == 0).sum())
    )
)

player_features["rating_gain_12m"] = (
    player_features["rating_end"] - player_features["rating_start"]
)

player_features["age_approx_2026"] = 2026 - player_features["Birth_Year"]

# Remove players without start/end ratings
player_features = player_features[
    player_features["rating_start"].notna() &
    player_features["rating_end"].notna()
].copy()

print(player_features.shape)
print(player_features.head(20).to_string(index=False))

In [ ]:
# Select exactly 1,000 players
# Use a reproducible random sample. stratified sample by rating band. This is better than pure random sampling because it avoids overrepresenting only low-rated players.

player_features["rating_band"] = pd.cut(
    player_features["rating_start"],
    bins=[0, 1000, 1200, 1400, 1600, 1800, 2000, 3000],
    labels=["<1000", "1000-1199", "1200-1399", "1400-1599", "1600-1799", "1800-1999", "2000+"],
    include_lowest=True
)

print(player_features["rating_band"].value_counts(dropna=False))

player_sample_1000 = (
    player_features
    .groupby("rating_band", group_keys=False, observed=True)
    .apply(lambda x: x.sample(
        n=min(len(x), max(1, round(len(x) / len(player_features) * 1000))),
        random_state=42
    ))
    .reset_index(drop=True)
)

# If rounding gives slightly more or less than 1000, adjust
if len(player_sample_1000) > 1000:
    player_sample_1000 = player_sample_1000.sample(n=1000, random_state=42)
elif len(player_sample_1000) < 1000:
    remaining = player_features[
        ~player_features["ID Number"].isin(player_sample_1000["ID Number"])
    ]
    needed = 1000 - len(player_sample_1000)
    extra = remaining.sample(n=min(needed, len(remaining)), random_state=42)
    player_sample_1000 = pd.concat([player_sample_1000, extra], ignore_index=True)

print(player_sample_1000.shape)
print(player_sample_1000["rating_band"].value_counts(dropna=False))



In [ ]:
# ============================================================
# Exploratory code cell 68
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

player_sample_1000_output_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "player_sample_1000.csv"

player_sample_1000.to_csv(player_sample_1000_output_path, index=False)


In [200]:
# For a balanced 1,000-player sample, use:
target_per_band = {
    "1400-1599": 600,
    "1600-1799": 250,
    "1800-1999": 100,
    "2000+": 50
}

sample_parts = []

for band, n in target_per_band.items():
    band_df = player_features[player_features["rating_band"] == band]
    actual_n = min(n, len(band_df))
    
    if actual_n > 0:
        sample_parts.append(
            band_df.sample(n=actual_n, random_state=42)
        )

balanced_sample_1000 = pd.concat(sample_parts, ignore_index=True)

print(balanced_sample_1000.shape)
print(balanced_sample_1000["rating_band"].value_counts())

(1000, 17)
rating_band
1400-1599    600
1600-1799    250
1800-1999    100
2000+         50
<1000          0
1000-1199      0
1200-1399      0
Name: count, dtype: int64


In [208]:
# ============================================================
# Exploratory code cell 70
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import  html5lib

ModuleNotFoundError: No module named 'html5lib'

In [1]:
#########################
import html5lib
import pandas as pd
import requests
from io import StringIO
from pathlib import Path
import time

fide_id = "25977890"   # Amartya
rating_type = 0        # 0 = Standard, 1 = Rapid, 2 = Blitz

start_period = "2024-01-01"
end_period = "2026-05-01"   # change to 2026-04-01 if needed

periods = pd.date_range(start_period, end_period, freq="MS")

output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "amartya_calculations"
output_dir.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

all_tables = []

for period in periods:
    period_str = period.strftime("%Y-%m-%d")
    
    url = (
        "https://ratings.fide.com/calculations.phtml"
        f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
    )
    
    print("\nReading:", period_str, url)
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        print("HTTP status:", response.status_code)
        response.raise_for_status()
        
        html = response.text
        
        # Save raw HTML for debugging
        html_path = output_dir / f"amartya_{period_str}_standard.html"
        html_path.write_text(html, encoding="utf-8")
        
        tables = pd.read_html(StringIO(html))
        print("Tables found:", len(tables))
        
        for i, table in enumerate(tables):
            temp = table.copy()
            temp["fide_id"] = fide_id
            temp["period"] = period_str
            temp["rating_type"] = rating_type
            temp["source_url"] = url
            temp["table_index"] = i
            
            all_tables.append(temp)
            
            print(f"Table {i} shape:", temp.shape)
            print(temp.head().to_string(index=False))
        
        time.sleep(2)
        
    except Exception as e:
        print("Failed:", period_str)
        print("Reason:", repr(e))

if all_tables:
    amartya_calc_raw = pd.concat(all_tables, ignore_index=True)
    
    output_csv = output_dir / "amartya_fide_calculations_raw.csv"
    amartya_calc_raw.to_csv(output_csv, index=False)
    
    print("\nSaved:", output_csv)
    print(amartya_calc_raw.shape)
    print(amartya_calc_raw.head(20).to_string(index=False))
else:
    print("\nNo tables extracted using simple requests/read_html.")


Reading: 2024-01-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-01-01&rating=0
HTTP status: 200
Failed: 2024-01-01
Reason: ValueError('No tables found')

Reading: 2024-02-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-02-01&rating=0
HTTP status: 200
Failed: 2024-02-01
Reason: ValueError('No tables found')

Reading: 2024-03-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-03-01&rating=0
HTTP status: 200
Failed: 2024-03-01
Reason: ValueError('No tables found')

Reading: 2024-04-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-04-01&rating=0
HTTP status: 200
Failed: 2024-04-01
Reason: ValueError('No tables found')

Reading: 2024-05-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-05-01&rating=0
HTTP status: 200
Failed: 2024-05-01
Reason: ValueError('No tables found')

Reading: 2024-06-01 https://ratings.fide.com/calculations.phtml?id_number=

In [3]:
# ============================================================
# Exploratory code cell 72
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

period_str = "2026-05-01"
fide_id = "25977890"
rating_type = 0

url = (
    "https://ratings.fide.com/calculations.phtml"
    f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
)

response = requests.get(url, headers=headers, timeout=30)
print(response.status_code)
print(response.text[:2000])

200
<!DOCTYPE html>
<html lang="en">
<head>
    <link rel="apple-touch-icon" href="/manifest/apple-touch-icon-iphone-60x60.png">
    <link rel="apple-touch-icon" sizes="60x60" href="/manifest/apple-touch-icon-ipad-76x76.png">
    <link rel="apple-touch-icon" sizes="114x114" href="/manifest/apple-touch-icon-iphone-retina-120x120.png">
    <link rel="apple-touch-icon" sizes="144x144" href="/manifest/apple-touch-icon-ipad-retina-152x152.png">

    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Amartya Saumitra Gupta IND Individual Calculations Chess Ratings FIDE</title>
    <link rel="stylesheet" type="text/css" href="/css/style.css">
    <link rel="stylesheet" type="text/css" href="/css/mobile.css">
	<link rel="stylesheet" type="text/css" href="/css/post_css.css">
	<link rel="stylesheet" type="text/css" href="/css/dt.css">

	<link rel="stylesheet" type="text/css" href="

In [5]:
# ============================================================
# Exploratory code cell 73
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

from pathlib import Path

output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "amartya_calculations"
output_dir.mkdir(parents=True, exist_ok=True)

html_path = output_dir / "debug_2026_05.html"
html_path.write_text(response.text, encoding="utf-8")

print("Saved HTML:", html_path)

Saved HTML: /Users/arunkumar/Downloads/fide_history/amartya_calculations/debug_2026_05.html


In [7]:
# ============================================================
# Exploratory code cell 74
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

html = response.text

keywords = [
    "Amartya",
    "Tournament",
    "Opponent",
    "Rating",
    "K",
    "Rc",
    "Change",
    "No data",
    "No tournaments"
]

for k in keywords:
    print(k, "=>", html.find(k))

Amartya => 612
Tournament => 4504
Opponent => -1
Rating => 669
K => -1
Rc => -1
Change => -1
No data => -1
No tournaments => -1


In [9]:
# ============================================================
# Exploratory code cell 75
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import requests
from io import StringIO
from pathlib import Path
import time

fide_id = "25977890"
rating_type = 0

periods = pd.date_range("2024-01-01", "2026-05-01", freq="MS")

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "amartya_calculations"
output_dir.mkdir(parents=True, exist_ok=True)

all_tables = []
month_status = []

for period in periods:
    period_str = period.strftime("%Y-%m-%d")

    url = (
        "https://ratings.fide.com/calculations.phtml"
        f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
    )

    print("\nReading:", period_str)

    try:
        response = requests.get(url, headers=headers, timeout=30)
        status_code = response.status_code
        html = response.text

        html_path = output_dir / f"amartya_{period_str}_standard.html"
        html_path.write_text(html, encoding="utf-8")

        try:
            tables = pd.read_html(StringIO(html), flavor="lxml")
        except ValueError:
            tables = []

        print("Tables found:", len(tables))

        month_status.append({
            "period": period_str,
            "status_code": status_code,
            "tables_found": len(tables),
            "html_file": str(html_path),
            "url": url
        })

        for i, table in enumerate(tables):
            temp = table.copy()
            temp["fide_id"] = fide_id
            temp["period"] = period_str
            temp["rating_type"] = rating_type
            temp["source_url"] = url
            temp["table_index"] = i
            all_tables.append(temp)

            print("Table", i, "shape:", temp.shape)
            print(temp.head().to_string(index=False))

        time.sleep(2)

    except Exception as e:
        print("Failed:", period_str, repr(e))

        month_status.append({
            "period": period_str,
            "status_code": None,
            "tables_found": 0,
            "html_file": None,
            "url": url,
            "error": repr(e)
        })

month_status_df = pd.DataFrame(month_status)

print(month_status_df.to_string(index=False))

if all_tables:
    amartya_calc_raw = pd.concat(all_tables, ignore_index=True)
    print(amartya_calc_raw.head(20).to_string(index=False))
else:
    print("No HTML tables extracted for any month.")


Reading: 2024-01-01
Tables found: 0

Reading: 2024-02-01
Tables found: 0

Reading: 2024-03-01
Tables found: 0

Reading: 2024-04-01
Tables found: 0

Reading: 2024-05-01
Tables found: 0

Reading: 2024-06-01
Tables found: 0

Reading: 2024-07-01
Tables found: 0

Reading: 2024-08-01
Tables found: 0

Reading: 2024-09-01
Tables found: 0

Reading: 2024-10-01
Tables found: 0

Reading: 2024-11-01
Tables found: 0

Reading: 2024-12-01
Tables found: 0

Reading: 2025-01-01
Tables found: 0

Reading: 2025-02-01
Tables found: 0

Reading: 2025-03-01
Tables found: 0

Reading: 2025-04-01
Tables found: 0

Reading: 2025-05-01
Tables found: 0

Reading: 2025-06-01
Tables found: 0

Reading: 2025-07-01
Tables found: 0

Reading: 2025-08-01
Tables found: 0

Reading: 2025-09-01
Tables found: 0

Reading: 2025-10-01
Tables found: 0

Reading: 2025-11-01
Tables found: 0

Reading: 2025-12-01
Tables found: 0

Reading: 2026-01-01
Tables found: 0

Reading: 2026-02-01
Tables found: 0

Reading: 2026-03-01
Tables found: 0



In [11]:
# ============================================================
# Exploratory code cell 76
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

from pathlib import Path
from bs4 import BeautifulSoup

html_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "amartya_calculations" / "amartya_2026-05-01_standard.html"

html = html_path.read_text(encoding="utf-8", errors="ignore")

soup = BeautifulSoup(html, "html.parser")

text = soup.get_text("\n", strip=True)

print(text[:5000])

Amartya Saumitra Gupta IND Individual Calculations Chess Ratings FIDE
International
Chess Federation
NEWS
RATINGS
CHAMPIONSHIP
CALENDAR
FIDE
DIRECTORY
PARTNERS
SHOP
CONTACTS
FIDE News
Chess news
Top
Top Federations
Tournaments
Titles
Transfers
Calculators
Download
FIDE Circuit
Women's Circuit '26-'27
Open Cycle '25-'26
Women’s Cycle '25-'26
Women’s Cycle '23-'25
All Tournaments
Main Events
About FIDE
Handbook
Documents
Clean Sport
Financial Reports
Officials
Commissions & Committees
Federations
Aff. Organizations
Aff. Members
Honourable Dignitaries
Chart
PARNTERS
FIDE100
CONTACTS
MAIN/NEWS
All News
FIDE News
Chess News
RATINGS
Top
Top Federations
Main Page / Search
Tournaments
Titles
Transfers
Calculators
Download
CHAMPIONSHIP
FIDE Circuit
Women's Circuit '26-'27
Open Cycle 2025-2026
Women’s Cycle 2025-2026
Women’s Cycle 2023-2025
CALENDAR
All Tournaments
Main Events
FIDE
About FIDE
Handbook
Documents
Clean Sport
Financial Reports
DIRECTORY
Officials
Commissions & Committees
Federation

In [13]:
# ============================================================
# Exploratory code cell 77
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

keywords = [
    "Amartya",
    "Saumitra",
    "Tournament",
    "Opponent",
    "Rc",
    "W",
    "We",
    "K",
    "Change",
    "Period",
    "Malakoff",
    "No data"
]

for k in keywords:
    print(k, "=>", text.find(k))

Amartya => 0
Saumitra => 8
Tournament => 215
Opponent => -1
Rc => -1
W => 103
We => -1
K => -1
Change => -1
Period => 1208
Malakoff => -1
No data => -1


In [19]:
!pip install playwright

  Using cached playwright-1.59.0-py3-none-macosx_11_0_arm64.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 MB 11.6 MB/s  0:00:031.6 MB/s eta 0:00:01:01
  Attempting uninstall: greenlet
    Found existing installation: greenlet 3.0.1
    Uninstalling greenlet-3.0.1:
      Successfully uninstalled greenlet-3.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [playwright]━━━━━━━ 2/3 [playwright]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [21]:
!python -m playwright install chromium

165.5 MiB [                    ] 0% 0.0s165.5 MiB [                    ] 0% 2017.1s165.5 MiB [                    ] 0% 1444.8s165.5 MiB [                    ] 0% 1643.7s165.5 MiB [                    ] 0% 1376.4s165.5 MiB [                    ] 0% 1718.7s165.5 MiB [                    ] 0% 1252.1s165.5 MiB [                    ] 0% 1153.7s165.5 MiB [                    ] 0% 1092.5s165.5 MiB [                    ] 0% 1040.3s165.5 MiB [                    ] 0% 982.3s165.5 MiB [                    ] 0% 933.8s165.5 MiB [                    ] 0% 892.9s165.5 MiB [                    ] 0% 852.4s165.5 MiB [                    ] 0% 817.4s165.5 MiB [                    ] 0% 778.8s165.5 MiB [                    ] 0% 744.8s165.5 MiB [                    ] 0% 713.3s165.5 MiB [                    ] 0% 688.0s165.5 MiB [                    ] 0% 662.0s165.5 MiB [                    ] 0% 639.5s165.5 MiB [                    ] 0% 619.5s165.5 MiB [                    ] 0% 602.1s165.5 MiB [                

In [1]:
# ============================================================
# Exploratory code cell 80
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

from playwright.sync_api import sync_playwright

print("Playwright installed successfully")

ModuleNotFoundError: No module named 'playwright'

In [17]:
# ============================================================
# Exploratory code cell 81
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

from playwright.sync_api import sync_playwright
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

fide_id = "25977890"
period_str = "2026-05-01"
rating_type = 0

url = (
    "https://ratings.fide.com/calculations.phtml"
    f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
)

output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "amartya_calculations"
output_dir.mkdir(parents=True, exist_ok=True)

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)  # keep visible for debugging
    page = browser.new_page()
    
    page.goto(url, wait_until="networkidle", timeout=60000)
    page.wait_for_timeout(5000)

    html = page.content()
    
    rendered_path = output_dir / "rendered_2026_05.html"
    rendered_path.write_text(html, encoding="utf-8")
    
    screenshot_path = output_dir / "rendered_2026_05.png"
    page.screenshot(path=str(screenshot_path), full_page=True)
    
    print("Saved rendered HTML:", rendered_path)
    print("Saved screenshot:", screenshot_path)

    try:
        tables = pd.read_html(StringIO(html))
        print("Tables found after rendering:", len(tables))
        for i, t in enumerate(tables):
            print("\nTable", i, t.shape)
            print(t.head(20).to_string(index=False))
    except Exception as e:
        print("Still no tables:", repr(e))

    browser.close()

ModuleNotFoundError: No module named 'playwright'

In [3]:
# ============================================================
# Exploratory code cell 82
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO
from playwright.async_api import async_playwright

fide_id = "25977890"
period_str = "2026-05-01"
rating_type = 0

url = (
    "https://ratings.fide.com/calculations.phtml"
    f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
)

output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "amartya_calculations"
output_dir.mkdir(parents=True, exist_ok=True)

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)  # visible browser
    page = await browser.new_page()

    await page.goto(url, wait_until="networkidle", timeout=60000)
    await page.wait_for_timeout(5000)

    html = await page.content()

    rendered_path = output_dir / "rendered_2026_05.html"
    rendered_path.write_text(html, encoding="utf-8")

    screenshot_path = output_dir / "rendered_2026_05.png"
    await page.screenshot(path=str(screenshot_path), full_page=True)

    print("Saved rendered HTML:", rendered_path)
    print("Saved screenshot:", screenshot_path)

    try:
        tables = pd.read_html(StringIO(html))
        print("Tables found after rendering:", len(tables))

        for i, t in enumerate(tables):
            print("\nTable", i, t.shape)
            print(t.head(30).to_string(index=False))

    except Exception as e:
        print("Still no tables:", repr(e))

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n", strip=True)

    text_path = output_dir / "rendered_2026_05.txt"
    text_path.write_text(text, encoding="utf-8")

    print("Saved text:", text_path)
    print(text[:5000])

    await browser.close()

Saved rendered HTML: /Users/arunkumar/Downloads/fide_history/amartya_calculations/rendered_2026_05.html
Saved screenshot: /Users/arunkumar/Downloads/fide_history/amartya_calculations/rendered_2026_05.png
Tables found after rendering: 2

Table 0 (13, 15)
                                   0                                     1                                     2                                     3                                     4                                     5                                     6                                     7                                     8                                     9                                     10                                    11  12  13  14
                                   Rc                                    Ro                                   NaN                                   NaN                                   NaN                                     w                                     n             

In [5]:
# ============================================================
# Exploratory code cell 83
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO

output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"/ "fide_history" / "amartya_calculations"

html_path = output_dir / "rendered_2026_05.html"
html = html_path.read_text(encoding="utf-8", errors="ignore")

# Read tables from rendered HTML
tables = pd.read_html(StringIO(html))

# Extract visible text for tournament metadata
soup = BeautifulSoup(html, "html.parser")
text_lines = [
    line.strip()
    for line in soup.get_text("\n", strip=True).split("\n")
    if line.strip()
]

print("Tables found:", len(tables))

Tables found: 2


In [7]:
# ============================================================
# Exploratory code cell 84
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def extract_event_metadata_from_text(text_lines):
    """
    Finds tournament blocks before each Rc/Ro/w/n/chg/K/K*chg table.
    Expected pattern:
    tournament_name
    CITY FED
    start_date
    end_date
    Rc
    Ro
    ...
    """

    events = []

    for i, line in enumerate(text_lines):
        if line == "Rc":
            # Look backward for two dates and event location
            if i >= 4:
                event_name = text_lines[i - 4]
                location_fed = text_lines[i - 3]
                start_date = text_lines[i - 2]
                end_date = text_lines[i - 1]

                events.append({
                    "tournament_name": event_name,
                    "event_location_fed": location_fed,
                    "event_start_date": start_date,
                    "event_end_date": end_date
                })

    return events

events = extract_event_metadata_from_text(text_lines)

print("Events found:", len(events))
for e in events:
    print(e)

Events found: 2
{'tournament_name': "Championnat International d'Echecs de Lyon Henri Rinck 2026", 'event_location_fed': 'LYON FRA', 'event_start_date': '2026-04-11', 'event_end_date': '2026-04-15'}
{'tournament_name': "11eme Open International d'Echecs de Noisiel 2026 - Open A (p2000 Elo)", 'event_location_fed': 'NOISIEL FRA', 'event_start_date': '2026-04-18', 'event_end_date': '2026-04-23'}


In [9]:
# ============================================================
# Exploratory code cell 85
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

def clean_fide_calculation_table(table, event_meta, fide_id, period, rating_type=0):
    """
    Cleans one FIDE calculation table from the rendered page.
    """

    df = table.copy()

    # Keep first 10 useful columns only
    df = df.iloc[:, :10].copy()

    # Row 0 = header, Row 1 = event summary, Row 2 usually blank
    summary = df.iloc[1]

    event_rc = pd.to_numeric(summary.iloc[0], errors="coerce")
    event_ro = pd.to_numeric(summary.iloc[1], errors="coerce")
    event_score = pd.to_numeric(summary.iloc[5], errors="coerce")
    event_games = pd.to_numeric(summary.iloc[6], errors="coerce")
    event_chg = pd.to_numeric(summary.iloc[7], errors="coerce")
    event_k = pd.to_numeric(summary.iloc[8], errors="coerce")
    event_kchg = pd.to_numeric(summary.iloc[9], errors="coerce")

    # Opponent rows start after row 2
    opponent_df = df.iloc[3:].copy()

    opponent_df.columns = [
        "opponent_name",
        "opponent_title_1",
        "opponent_title_2",
        "opponent_rating_raw",
        "opponent_fed",
        "score",
        "n",
        "chg",
        "k",
        "k_chg"
    ]

    # Remove footnote rows and empty rows
    opponent_df = opponent_df[
        opponent_df["opponent_name"].notna()
    ].copy()

    opponent_df = opponent_df[
        ~opponent_df["opponent_name"].astype(str).str.contains("Rating difference", case=False, na=False)
    ].copy()

    # Keep only rows where opponent federation looks valid
    opponent_df["opponent_fed"] = opponent_df["opponent_fed"].astype(str).str.strip()

    opponent_df = opponent_df[
        opponent_df["opponent_fed"].str.match(r"^[A-Z]{3}$", na=False)
    ].copy()

    # Combine title fields
    opponent_df["opponent_title"] = (
        opponent_df[["opponent_title_1", "opponent_title_2"]]
        .astype(str)
        .replace("nan", "", regex=False)
        .agg(" ".join, axis=1)
        .str.strip()
    )

    opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan)

    # Rating-difference flag if rating has *
    opponent_df["rating_difference_over_400_flag"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.contains(r"\*", regex=True, na=False)
    )

    # Clean opponent rating
    opponent_df["opponent_rating"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.replace("*", "", regex=False)
        .str.strip()
    )

    opponent_df["opponent_rating"] = pd.to_numeric(
        opponent_df["opponent_rating"],
        errors="coerce"
    )

    # Numeric columns
    for col in ["score", "n", "chg", "k", "k_chg"]:
        opponent_df[col] = pd.to_numeric(opponent_df[col], errors="coerce")

    # Add event-level fields
    opponent_df["fide_id"] = str(fide_id)
    opponent_df["period"] = period
    opponent_df["rating_type"] = rating_type

    opponent_df["tournament_name"] = event_meta.get("tournament_name")
    opponent_df["event_location_fed"] = event_meta.get("event_location_fed")
    opponent_df["event_start_date"] = event_meta.get("event_start_date")
    opponent_df["event_end_date"] = event_meta.get("event_end_date")

    opponent_df["event_rc"] = event_rc
    opponent_df["event_ro"] = event_ro
    opponent_df["event_score"] = event_score
    opponent_df["event_games"] = event_games
    opponent_df["event_chg"] = event_chg
    opponent_df["event_k"] = event_k
    opponent_df["event_k_chg"] = event_kchg

    # Keep useful columns
    keep_cols = [
        "fide_id", "period", "rating_type",
        "tournament_name", "event_location_fed",
        "event_start_date", "event_end_date",
        "event_rc", "event_ro", "event_score", "event_games",
        "event_chg", "event_k", "event_k_chg",
        "opponent_name", "opponent_title", "opponent_rating",
        "rating_difference_over_400_flag",
        "opponent_fed", "score", "n", "chg", "k", "k_chg"
    ]

    return opponent_df[keep_cols].reset_index(drop=True)

In [11]:
# ============================================================
# Exploratory code cell 86
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

fide_id = "25977890"
period = "2026-05-01"
rating_type = 0

clean_tables = []

for i, table in enumerate(tables):
    if i < len(events):
        event_meta = events[i]
    else:
        event_meta = {}

    clean_one = clean_fide_calculation_table(
        table=table,
        event_meta=event_meta,
        fide_id=fide_id,
        period=period,
        rating_type=rating_type
    )

    clean_tables.append(clean_one)

amartya_may2026_calc = pd.concat(clean_tables, ignore_index=True)

print(amartya_may2026_calc.to_string(index=False))
print(amartya_may2026_calc.shape)

 fide_id     period  rating_type                                                        tournament_name event_location_fed event_start_date event_end_date  event_rc  event_ro  event_score  event_games  event_chg  event_k  event_k_chg             opponent_name opponent_title  opponent_rating  rating_difference_over_400_flag opponent_fed  score  n   chg  k  k_chg
25977890 2026-05-01            0            Championnat International d'Echecs de Lyon Henri Rinck 2026           LYON FRA       2026-04-11     2026-04-15      2096      2084          6.5            9       2.26       38        85.88         Hanriot, Frederik            NaN           1684.0                             True          FRA    1.0  1  0.08 38   3.04
25977890 2026-05-01            0            Championnat International d'Echecs de Lyon Henri Rinck 2026           LYON FRA       2026-04-11     2026-04-15      2096      2084          6.5            9       2.26       38        85.88        Bazzazi, Abdulaziz            N

In [13]:
# ============================================================
# Exploratory code cell 87
# Purpose:
# - This cell is part of the original exploration workflow.
# - It is retained for transparency and project reproducibility.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

output_csv = output_dir / "amartya_2026_05_standard_clean.csv"

amartya_may2026_calc.to_csv(output_csv, index=False)

print("Saved:", output_csv)

Saved: /Users/arunkumar/Downloads/fide_history/amartya_calculations/amartya_2026_05_standard_clean.csv
